# Notebook 2 — Suzlon Energy: Silver → Gold

**Purpose:** Build curated Gold tables for KPI reporting and business insights. The notebook creates monthly/site/segment aggregates, uses `ROLLUP`/`GROUPING SETS`, and implements the required ranking, moving-average and lag window operations.

In [0]:
from pyspark.sql import functions as F
CATALOG="workspace"; SCHEMA="default"
def t(name): return f"{CATALOG}.{SCHEMA}.{name}"


## 1. Monthly generation mart

Grain: site × year × month. This is the main operations mart for availability and generation trends.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_monthly_generation")} AS
SELECT
    g.site_id,
    s.site_name,
    s.state,
    s.region,
    d.year,
    d.month,
    ROUND(SUM(g.energy_kwh)/1000000,2) AS energy_gwh,
    ROUND(AVG(g.availability_pct),2) AS avg_availability_pct,
    ROUND(SUM(g.downtime_hours),2) AS downtime_hours
FROM {t("silver_fact_generation")} g
JOIN {t("silver_dim_site")} s ON g.site_id=s.site_id
JOIN {t("silver_dim_date")} d ON g.date_id=d.date_id
GROUP BY g.site_id,s.site_name,s.state,s.region,d.year,d.month
""")
display(spark.table(t("gold_monthly_generation")).orderBy("site_id","year","month").limit(20))


site_id,site_name,state,region,year,month,energy_gwh,avg_availability_pct,downtime_hours
1,WindSite_0001,Odisha,East,2025,1,1.64,96.33,54.61
1,WindSite_0001,Odisha,East,2025,2,1.45,95.90,55.13
1,WindSite_0001,Odisha,East,2025,3,1.56,96.23,56.06
1,WindSite_0001,Odisha,East,2025,4,1.63,96.13,55.74
1,WindSite_0001,Odisha,East,2025,5,1.74,95.79,62.58
1,WindSite_0001,Odisha,East,2025,6,1.72,95.83,60.09
1,WindSite_0001,Odisha,East,2025,7,1.84,96.25,55.77
1,WindSite_0001,Odisha,East,2025,8,1.91,96.05,58.78
1,WindSite_0001,Odisha,East,2025,9,1.74,96.28,53.57
1,WindSite_0001,Odisha,East,2025,10,1.74,96.43,53.12


## 2. Multi-level rollup with ROLLUP

`ROLLUP` is appropriate for the site hierarchy because management may want monthly detail, site totals, state totals and a grand total.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_generation_rollup")} AS
SELECT
    s.state,
    s.site_id,
    d.year,
    d.month,
    ROUND(SUM(g.energy_kwh)/1000000,2) AS energy_gwh,
    GROUPING(s.state) AS state_grouped,
    GROUPING(s.site_id) AS site_grouped
FROM {t("silver_fact_generation")} g
JOIN {t("silver_dim_site")} s ON g.site_id=s.site_id
JOIN {t("silver_dim_date")} d ON g.date_id=d.date_id
GROUP BY ROLLUP(s.state,s.site_id,d.year,d.month)
""")
display(spark.table(t("gold_generation_rollup")).limit(30))


state,site_id,year,month,energy_gwh,state_grouped,site_grouped
Madhya Pradesh,96,2025,3,2.25,0,0
Kerala,43,2025,10,6.51,0,0
Gujarat,154,2025,3,2.80,0,0
Gujarat,154,2025,10,2.97,0,0
Gujarat,154,2025,11,2.84,0,0
Madhya Pradesh,158,2025,11,4.76,0,0
Tamil Nadu,270,2025,4,1.78,0,0
Telangana,120,2025,8,4.06,0,0
Gujarat,285,2025,3,3.34,0,0
Rajasthan,144,2025,6,5.04,0,0


## 3. Customer-segment and period GROUPING SETS

`GROUPING SETS` is used because the dashboard needs selected combinations rather than every possible hierarchy combination.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_order_grouping_sets")} AS
SELECT
    c.segment,
    d.year,
    d.month,
    ROUND(SUM(o.order_value_inr)/10000000,2) AS order_value_crore,
    GROUPING(c.segment) AS segment_grouped,
    GROUPING(d.year) AS year_grouped,
    GROUPING(d.month) AS month_grouped
FROM {t("silver_fact_order")} o
JOIN {t("silver_dim_customer")} c ON o.customer_id=c.customer_id
JOIN {t("silver_dim_date")} d ON o.date_id=d.date_id
GROUP BY GROUPING SETS (
    (c.segment,d.year,d.month),
    (c.segment,d.year),
    (d.year,d.month),
    ()
)
""")
display(spark.table(t("gold_order_grouping_sets")).orderBy("year","month").limit(50))


segment,year,month,order_value_crore,segment_grouped,year_grouped,month_grouped
null,null,null,1273619.77,1,1,1
Retail,2025,null,102693.71,0,0,1
C&I,2025,null,505009.80,0,0,1
IPP,2025,null,420212.39,0,0,1
PSU,2025,null,245703.87,0,0,1
null,2025,1,104769.02,1,0,0
IPP,2025,1,34634.62,0,0,0
Retail,2025,1,8675.90,0,0,0
PSU,2025,1,20408.90,0,0,0
C&I,2025,1,41049.61,0,0,0


## 4. Window insight — ranking within each site

Business question: **Which turbines are the strongest generators within each site?**

The partition is the site, so each site's turbines are ranked against one another.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_turbine_rank")} AS
WITH turbine_totals AS (
  SELECT site_id,turbine_id,
         ROUND(SUM(energy_kwh)/1000000,2) AS energy_gwh
  FROM {t("silver_fact_generation")}
  GROUP BY site_id,turbine_id
)
SELECT *,
       RANK() OVER (PARTITION BY site_id ORDER BY energy_gwh DESC) AS turbine_rank_in_site
FROM turbine_totals
""")
display(spark.table(t("gold_turbine_rank")).filter("turbine_rank_in_site <= 3").orderBy("site_id","turbine_rank_in_site").limit(30))


site_id,turbine_id,energy_gwh,turbine_rank_in_site
1,528,14.42,1
1,39,5.81,2
2,839,14.37,1
2,358,11.47,2
2,451,11.23,3
3,699,14.21,1
3,186,11.57,2
3,836,11.43,3
4,770,13.45,1
4,360,10.34,2


## 5. Window insight — three-month moving average

Business question: **Is a site's generation improving or deteriorating beyond short-term noise?**

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_generation_moving_avg")} AS
SELECT *,
       ROUND(
         AVG(energy_gwh) OVER (
           PARTITION BY site_id
           ORDER BY year,month
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
         ),2
       ) AS moving_avg_3m_gwh
FROM {t("gold_monthly_generation")}
""")
display(spark.table(t("gold_generation_moving_avg")).orderBy("site_id","year","month").limit(30))


site_id,site_name,state,region,year,month,energy_gwh,avg_availability_pct,downtime_hours,moving_avg_3m_gwh
1,WindSite_0001,Odisha,East,2025,1,1.64,96.33,54.61,1.64
1,WindSite_0001,Odisha,East,2025,2,1.45,95.90,55.13,1.55
1,WindSite_0001,Odisha,East,2025,3,1.56,96.23,56.06,1.55
1,WindSite_0001,Odisha,East,2025,4,1.63,96.13,55.74,1.55
1,WindSite_0001,Odisha,East,2025,5,1.74,95.79,62.58,1.64
1,WindSite_0001,Odisha,East,2025,6,1.72,95.83,60.09,1.70
1,WindSite_0001,Odisha,East,2025,7,1.84,96.25,55.77,1.77
1,WindSite_0001,Odisha,East,2025,8,1.91,96.05,58.78,1.82
1,WindSite_0001,Odisha,East,2025,9,1.74,96.28,53.57,1.83
1,WindSite_0001,Odisha,East,2025,10,1.74,96.43,53.12,1.80


## 6. Window insight — month-over-month LAG

Business question: **Which sites experienced a significant month-over-month change in generation?**

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_generation_lag")} AS
SELECT *,
       LAG(energy_gwh) OVER (
         PARTITION BY site_id
         ORDER BY year,month
       ) AS previous_month_energy_gwh
FROM {t("gold_monthly_generation")}
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_generation_mom")} AS
SELECT *,
       ROUND(100*(energy_gwh-previous_month_energy_gwh)/NULLIF(previous_month_energy_gwh,0),2) AS mom_change_pct
FROM {t("gold_generation_lag")}
""")
display(spark.table(t("gold_generation_mom")).filter("previous_month_energy_gwh IS NOT NULL").orderBy(F.abs(F.col("mom_change_pct")).desc()).limit(20))


site_id,site_name,state,region,year,month,energy_gwh,avg_availability_pct,downtime_hours,previous_month_energy_gwh,mom_change_pct
134,WindSite_0134,Madhya Pradesh,Central,2025,3,0.40,97.31,19.99,0.33,21.21
193,WindSite_0193,Madhya Pradesh,Central,2025,3,3.53,96.81,94.76,2.97,18.86
122,WindSite_0122,Karnataka,South,2025,3,0.71,97.36,19.63,0.60,18.33
69,WindSite_0069,Maharashtra,West,2025,3,1.18,96.45,52.77,1.00,18.00
134,WindSite_0134,Madhya Pradesh,Central,2025,2,0.33,95.87,27.74,0.40,-17.50
71,WindSite_0071,Telangana,South,2025,3,0.47,96.62,25.10,0.40,17.50
40,WindSite_0040,Kerala,South,2025,5,0.68,92.87,53.02,0.58,17.24
265,WindSite_0265,Kerala,South,2025,3,2.07,96.88,69.47,1.77,16.95
224,WindSite_0224,Telangana,South,2025,3,3.60,97.17,84.02,3.08,16.88
235,WindSite_0235,Rajasthan,North,2025,2,1.04,96.87,21.02,1.25,-16.80


## 7. KPI aggregates

These tables are designed for dashboard trends and comparisons rather than one-number summaries.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_kpi_monthly")} AS

WITH months AS (
    SELECT DISTINCT year, month
    FROM {t("silver_dim_date")}
),

orders AS (
    SELECT
        d.year,
        d.month,
        ROUND(SUM(o.order_value_inr)/10000000, 2)
            AS order_booking_value_crore
    FROM {t("silver_fact_order")} o
    JOIN {t("silver_dim_date")} d
        ON o.date_id = d.date_id
    GROUP BY d.year, d.month
),

contracts AS (
    SELECT
        d.year,
        d.month,
        ROUND(100 * AVG(c.renewed_flag), 2)
            AS contract_renewal_rate_pct
    FROM {t("silver_fact_contract")} c
    JOIN {t("silver_dim_date")} d
        ON c.date_id = d.date_id
    GROUP BY d.year, d.month
),

generation AS (
    SELECT
        d.year,
        d.month,
        ROUND(AVG(g.availability_pct), 2)
            AS fleet_availability_pct,
        ROUND(SUM(g.energy_kwh)/1000000, 2)
            AS energy_gwh
    FROM {t("silver_fact_generation")} g
    JOIN {t("silver_dim_date")} d
        ON g.date_id = d.date_id
    GROUP BY d.year, d.month
),

service AS (
    SELECT
        d.year,
        d.month,
        ROUND(SUM(s.downtime_hours), 2)
            AS service_downtime_hours,
        COUNT(*) AS service_ticket_count
    FROM {t("silver_fact_service")} s
    JOIN {t("silver_dim_date")} d
        ON s.date_id = d.date_id
    GROUP BY d.year, d.month
)

SELECT
    m.year,
    m.month,
    COALESCE(o.order_booking_value_crore, 0)
        AS order_booking_value_crore,
    COALESCE(c.contract_renewal_rate_pct, 0)
        AS contract_renewal_rate_pct,
    COALESCE(g.fleet_availability_pct, 0)
        AS fleet_availability_pct,
    COALESCE(g.energy_gwh, 0)
        AS energy_gwh,
    COALESCE(s.service_downtime_hours, 0)
        AS service_downtime_hours,
    COALESCE(s.service_ticket_count, 0)
        AS service_ticket_count
FROM months m
LEFT JOIN orders o
    ON m.year=o.year AND m.month=o.month
LEFT JOIN contracts c
    ON m.year=c.year AND m.month=c.month
LEFT JOIN generation g
    ON m.year=g.year AND m.month=g.month
LEFT JOIN service s
    ON m.year=s.year AND m.month=s.month
""")

display(
    spark.table(t("gold_kpi_monthly"))
         .orderBy("year", "month")
)

year,month,order_booking_value_crore,contract_renewal_rate_pct,fleet_availability_pct,energy_gwh,service_downtime_hours,service_ticket_count
2025,1,104769.02,76.3,96.38,766.56,11974.78,2525
2025,2,96472.56,73.77,96.38,675.69,10808.19,2284
2025,3,108868.59,70.89,96.38,758.58,11527.84,2545
2025,4,108247.91,74.92,96.39,768.19,11347.77,2397
2025,5,107707.24,73.62,96.40,822.75,12030.76,2542
2025,6,102710.93,74.0,96.38,823.65,11211.43,2472
2025,7,108429.32,74.84,96.38,885.06,12372.38,2606
2025,8,108394.70,70.45,96.38,892.41,11901.57,2574
2025,9,107259.51,72.79,96.39,836.99,11702.05,2490
2025,10,111531.54,75.28,96.37,822.51,11824.11,2543


## Important aggregation note

The monthly KPI table above joins several facts by date. For dashboard KPIs where independent fact grains could multiply rows, use the dedicated Gold tables below instead of this convenience table. The final dashboard queries therefore calculate each KPI from its appropriate Gold/Silver source at the correct grain.

In [0]:
# Safer dedicated KPI tables
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_monthly_orders")} AS
SELECT d.year,d.month,ROUND(SUM(o.order_value_inr)/10000000,2) order_booking_value_crore
FROM {t("silver_fact_order")} o JOIN {t("silver_dim_date")} d ON o.date_id=d.date_id
GROUP BY d.year,d.month
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_monthly_contracts")} AS
SELECT d.year,d.month,ROUND(100*AVG(c.renewed_flag),2) renewal_rate_pct
FROM {t("silver_fact_contract")} c JOIN {t("silver_dim_date")} d ON c.date_id=d.date_id
GROUP BY d.year,d.month
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_monthly_availability")} AS
SELECT d.year,d.month,ROUND(AVG(g.availability_pct),2) fleet_availability_pct
FROM {t("silver_fact_generation")} g JOIN {t("silver_dim_date")} d ON g.date_id=d.date_id
GROUP BY d.year,d.month
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {t("gold_monthly_service")} AS
SELECT d.year,d.month,COUNT(*) ticket_count,ROUND(SUM(downtime_hours),2) downtime_hours
FROM {t("silver_fact_service")} sv JOIN {t("silver_dim_date")} d ON sv.date_id=d.date_id
GROUP BY d.year,d.month
""")


DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

Additional Gold aggregates for dashboard KPIs
These curated tables pre-aggregate contract renewal performance by customer segment and service downtime by ticket type. They allow the Databricks dashboard to query only Gold-zone data, while preserving clear business grains for comparison.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.gold_contract_renewal_segment AS
SELECT
    c.segment AS customer_segment,
    COUNT(*) AS contract_count,
    ROUND(100.0 * AVG(ct.renewed_flag), 2) AS renewal_rate_pct
FROM workspace.default.silver_fact_contract ct
JOIN workspace.default.silver_dim_customer c
    ON ct.customer_id = c.customer_id
GROUP BY c.segment
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.default.gold_service_ticket_type AS
SELECT
    ticket_type,
    COUNT(*) AS ticket_count,
    ROUND(SUM(downtime_hours), 2) AS downtime_hours,
    ROUND(AVG(downtime_hours), 2) AS avg_downtime_hours
FROM workspace.default.silver_fact_service
GROUP BY ticket_type
""")

print("Dashboard Gold tables created successfully.")

Dashboard Gold tables created successfully.
